# Window/Level Data Analysis: Original vs Synthetic Mammograms

This notebook performs W/L normalization on original NPZ files and provides visual comparison with synthetic images.

## Goals:
1. Convert original NPZ files to W/L-normalized PNG files
2. Visual comparison: Original NPZ (raw) vs W/L-normalized PNG
3. Visual comparison: W/L-normalized PNG vs Synthetic PNG
4. Verify pixel value alignment after W/L transformation

**IMPORTANT: All original files are READ-ONLY. New files saved to scratch folder.**


## Part 1: Setup and Paths


In [ ]:
import os
import glob
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from tqdm import tqdm

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# READ-ONLY paths (original data - DO NOT MODIFY)
metadata_path = '/hpcstor6/scratch01/a/a.kanamarlapudi001/datasets/2d_resized_512/metadata'
original_npz_path = '/hpcstor6/scratch01/a/a.kanamarlapudi001/datasets/2d_resized_512/images'
synthetic_path = '/raid/mpsych/OMAMA/DATA/data/train'

# Output path for W/L-normalized files (NEW files created here)
output_path = '/hpcstor6/scratch01/a/a.kanamarlapudi001/synthetic_test/original_wl_normalized'
os.makedirs(output_path, exist_ok=True)

print("=== READ-ONLY DATA PATHS ===")
print(f"Metadata (READ-ONLY): {metadata_path}")
print(f"Original NPZ (READ-ONLY): {original_npz_path}")
print(f"Synthetic PNG (READ-ONLY): {synthetic_path}")
print(f"\nOutput path (NEW files): {output_path}")
print("\n✓ All original files will remain untouched!")


## Part 2: Verify Data Availability


In [ ]:
# Check data availability
metadata_files = glob.glob(os.path.join(metadata_path, '*.json'))
original_npz_files = glob.glob(os.path.join(original_npz_path, '*.npz'))
synthetic_files = glob.glob(os.path.join(synthetic_path, '*.png'))

print(f"Found {len(metadata_files)} metadata JSON files")
print(f"Found {len(original_npz_files)} original NPZ files")
print(f"Found {len(synthetic_files)} synthetic PNG files")

# Check for matching files
npz_basenames = set(os.path.basename(f).replace('.npz', '') for f in original_npz_files)
meta_basenames = set(os.path.basename(f).replace('.json', '') for f in metadata_files)
matching_files = npz_basenames.intersection(meta_basenames)

print(f"\nFiles with both NPZ and metadata: {len(matching_files)}")
print(f"Sample matching files: {list(matching_files)[:5]}")

if len(matching_files) == 0:
    print("\n⚠ WARNING: No matching NPZ and metadata files found!")
else:
    print("\n✓ Data verification complete!")


## Part 3: Window/Level Normalization Function


In [ ]:
def apply_window_level(npz_data, window_center, window_width):
    """
    Apply Window/Level transformation to NPZ data.
    
    Args:
        npz_data: Raw NPZ image data (12-bit range)
        window_center: Window center value from metadata
        window_width: Window width value from metadata
    
    Returns:
        Normalized image (8-bit range, 0-255)
    """
    # Calculate window bounds
    wmin = window_center - window_width / 2
    wmax = window_center + window_width / 2
    
    # Apply W/L transformation
    clipped = np.clip(npz_data, wmin, wmax)
    normalized = ((clipped - wmin) / (wmax - wmin)) * 255
    
    return normalized.astype(np.uint8)

def load_metadata(metadata_file):
    """
    Load Window/Level values from metadata JSON file.
    
    Returns:
        tuple: (window_center, window_width)
    """
    try:
        with open(metadata_file, 'r') as f:
            data = json.load(f)
        
        wc = data.get('WindowCenter', 2730.0)  # Default from analysis
        ww = data.get('WindowWidth', 1132.2)   # Default from analysis
        
        # Handle lists (take first value)
        if isinstance(wc, list):
            wc = wc[0]
        if isinstance(ww, list):
            ww = ww[0]
            
        return float(wc), float(ww)
    except Exception as e:
        print(f"Warning: Could not load metadata from {metadata_file}: {e}")
        return 2730.0, 1132.2  # Default values

print("✓ W/L normalization functions defined")


## Part 4: Convert Original NPZ to W/L-Normalized PNG


In [ ]:
# Process first 100 files for analysis (adjust as needed)
num_files_to_process = 100
matching_list = list(matching_files)[:num_files_to_process]

print(f"Processing {len(matching_list)} files for W/L normalization...")
print("This will take a few minutes...")

conversion_stats = {
    'processed': 0,
    'failed': 0,
    'window_centers': [],
    'window_widths': [],
    'original_means': [],
    'normalized_means': []
}

for filename in tqdm(matching_list, desc="Converting NPZ to W/L PNG"):
    try:
        # File paths
        npz_file = os.path.join(original_npz_path, f"{filename}.npz")
        meta_file = os.path.join(metadata_path, f"{filename}.json")
        output_file = os.path.join(output_path, f"{filename}.png")
        
        # Skip if already converted
        if os.path.exists(output_file):
            continue
        
        # Load NPZ data
        with np.load(npz_file, allow_pickle=True) as data:
            npz_img = data['data']
        
        # Load metadata
        wc, ww = load_metadata(meta_file)
        
        # Apply W/L normalization
        normalized_img = apply_window_level(npz_img, wc, ww)
        
        # Save as PNG
        cv2.imwrite(output_file, normalized_img)
        
        # Collect statistics
        conversion_stats['processed'] += 1
        conversion_stats['window_centers'].append(wc)
        conversion_stats['window_widths'].append(ww)
        conversion_stats['original_means'].append(np.mean(npz_img))
        conversion_stats['normalized_means'].append(np.mean(normalized_img))
        
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        conversion_stats['failed'] += 1

print(f"\n=== CONVERSION COMPLETE ===")
print(f"Successfully processed: {conversion_stats['processed']} files")
print(f"Failed: {conversion_stats['failed']} files")
print(f"Output directory: {output_path}")


## Part 5: Conversion Statistics


In [ ]:
if conversion_stats['processed'] > 0:
    print("=== CONVERSION STATISTICS ===")
    print(f"\nWindow/Level Values:")
    print(f"  WindowCenter: {np.mean(conversion_stats['window_centers']):.1f} ± {np.std(conversion_stats['window_centers']):.1f}")
    print(f"  WindowWidth:  {np.mean(conversion_stats['window_widths']):.1f} ± {np.std(conversion_stats['window_widths']):.1f}")
    
    print(f"\nPixel Value Changes:")
    print(f"  Original NPZ mean: {np.mean(conversion_stats['original_means']):.1f} ± {np.std(conversion_stats['original_means']):.1f}")
    print(f"  Normalized PNG mean: {np.mean(conversion_stats['normalized_means']):.1f} ± {np.std(conversion_stats['normalized_means']):.1f}")
    
    mean_reduction = np.mean(conversion_stats['original_means']) - np.mean(conversion_stats['normalized_means'])
    print(f"  Mean reduction: {mean_reduction:.1f} points")
    
    print(f"\n✓ W/L normalization successfully applied!")
else:
    print("⚠ No files were successfully processed.")


## Part 6: Visual Comparison - Original NPZ vs W/L-Normalized PNG


In [ ]:
# Get converted files for comparison
converted_files = glob.glob(os.path.join(output_path, '*.png'))

if len(converted_files) >= 10:
    # Select 10 random samples
    random.seed(42)
    sample_files = random.sample(converted_files, 10)
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 10, figsize=(20, 6))
    
    for idx, converted_file in enumerate(sample_files):
        filename = os.path.basename(converted_file).replace('.png', '')
        
        # Load original NPZ (raw)
        npz_file = os.path.join(original_npz_path, f"{filename}.npz")
        with np.load(npz_file, allow_pickle=True) as data:
            original_npz = data['data']
        
        # Load W/L-normalized PNG
        normalized_png = cv2.imread(converted_file, cv2.IMREAD_GRAYSCALE)
        
        # Display original (raw)
        axes[0, idx].imshow(original_npz, cmap='gray', vmin=0, vmax=4095)
        axes[0, idx].set_title(f'Original NPZ #{idx+1}\nRaw: [{original_npz.min():.0f}, {original_npz.max():.0f}]', 
                              fontsize=8)
        axes[0, idx].axis('off')
        
        # Display normalized
        axes[1, idx].imshow(normalized_png, cmap='gray')
        axes[1, idx].set_title(f'W/L Normalized #{idx+1}\nRange: [{normalized_png.min()}, {normalized_png.max()}]', 
                               fontsize=8)
        axes[1, idx].axis('off')
    
    plt.suptitle('Comparison: Original NPZ (Raw) → W/L-Normalized PNG', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("✓ Visual comparison: Original NPZ vs W/L-Normalized PNG complete!")
else:
    print(f"⚠ Not enough converted files for comparison ({len(converted_files)} < 10)")


## Part 7: Visual Comparison - W/L-Normalized PNG vs Synthetic PNG


In [ ]:
if len(converted_files) >= 10 and len(synthetic_files) >= 10:
    # Select 10 random samples from each
    random.seed(123)
    sample_converted = random.sample(converted_files, 10)
    sample_synthetic = random.sample(synthetic_files, 10)
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 10, figsize=(20, 6))
    
    for idx in range(10):
        # Load W/L-normalized PNG
        normalized_img = cv2.imread(sample_converted[idx], cv2.IMREAD_GRAYSCALE)
        
        # Load synthetic PNG
        synthetic_img = cv2.imread(sample_synthetic[idx], cv2.IMREAD_GRAYSCALE)
        
        # Display W/L-normalized
        axes[0, idx].imshow(normalized_img, cmap='gray')
        axes[0, idx].set_title(f'W/L Normalized #{idx+1}\nRange: [{normalized_img.min()}, {normalized_img.max()}]', 
                               fontsize=8)
        axes[0, idx].axis('off')
        
        # Display synthetic
        axes[1, idx].imshow(synthetic_img, cmap='gray')
        axes[1, idx].set_title(f'Synthetic PNG #{idx+1}\nRange: [{synthetic_img.min()}, {synthetic_img.max()}]', 
                               fontsize=8)
        axes[1, idx].axis('off')
    
    plt.suptitle('Comparison: W/L-Normalized PNG → Synthetic PNG', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("✓ Visual comparison: W/L-Normalized PNG vs Synthetic PNG complete!")
else:
    print(f"⚠ Not enough files for comparison (converted: {len(converted_files)}, synthetic: {len(synthetic_files)})")


## Part 8: Summary and Next Steps


In [ ]:
print("=== WINDOW/LEVEL DATA ANALYSIS SUMMARY ===")
print(f"\nFiles processed: {conversion_stats['processed']}")
print(f"Output directory: {output_path}")
print(f"\nOriginal files remain untouched: ✓")

if conversion_stats['processed'] > 0:
    print(f"\n=== KEY FINDINGS ===")
    print(f"1. W/L normalization successfully applied")
    print(f"2. Pixel value range: 12-bit NPZ → 8-bit PNG")
    print(f"3. Mean pixel reduction: {np.mean(conversion_stats['original_means']) - np.mean(conversion_stats['normalized_means']):.1f} points")
    
    print(f"\n=== NEXT STEPS ===")
    print(f"1. ✓ W/L-normalized files created in: {output_path}")
    print(f"2. → Train classifier with W/L-normalized originals vs synthetic PNGs")
    print(f"3. → Expected result: Accuracy should drop from 99%+ to ~50-70%")
    print(f"4. → This would indicate realistic medical content distinguishability")
    
    print(f"\n=== READY FOR CLASSIFIER TRAINING ===")
    print(f"W/L-normalized original PNGs: {output_path}")
    print(f"Synthetic PNGs: {synthetic_path}")
    
else:
    print(f"\n⚠ No files were successfully processed. Check error messages above.")

print(f"\n=== ANALYSIS COMPLETE ===")
